In [1]:
print("a")

a


In [6]:
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from difflib import SequenceMatcher


# =========================
# Config
# =========================
INPUT_FOLDER = r"Files"   # <-- change this
OUTPUT_FILE = r"excel_column_metadata_output.xlsx"  # <-- change this

SUPPORTED_EXTENSIONS = {".xlsx", ".xlsm", ".xlsb", ".xls"}

# Optional: expand or adjust synonym mappings for your business context
SYNONYM_MAP = {
    "qty": "quantity",
    "q'ty": "quantity",
    "uom": "unit of measure",
    "unitofmeasure": "unit of measure",
    "unitmeasure": "unit of measure",
    "desc": "description",
    "descr": "description",
    "prod": "product",
    "prd": "product",
    "cust": "customer",
    "mfr": "manufacturer",
    "manuf": "manufacturer",
    "mnf": "manufacturer",
    "mpn": "manufacturer part number",
    "pn": "part number",
    "itemno": "item number",
    "itm": "item",
    "amt": "amount",
    "addr": "address",
    "dt": "date",
    "num": "number",
    "#": "number",
    "id": "id",
    "sku": "sku",
    "po": "purchase order",
    "gl": "general ledger",
    "dept": "department",
    "cat": "category",
    "subcat": "subcategory",
    "pkg": "packaging",
    "ea": "each",
}


# =========================
# Text normalization helpers
# =========================
def clean_column_name(col_name: str) -> str:
    """
    Basic cleanup of a raw column name into a normalized text string.
    """
    if pd.isna(col_name):
        return ""

    s = str(col_name).strip().lower()

    # Replace separators with spaces
    s = re.sub(r"[_\-/\\|]+", " ", s)

    # Split camelCase / PascalCase
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", s)

    # Keep alphanumeric and spaces
    s = re.sub(r"[^a-z0-9\s#]", " ", s)

    # Collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    return s


def apply_synonyms(text: str) -> str:
    """
    Replace token-level known abbreviations / common aliases.
    """
    if not text:
        return ""

    tokens = text.split()
    expanded = []

    for token in tokens:
        expanded_value = SYNONYM_MAP.get(token, token)
        expanded.append(expanded_value)

    text2 = " ".join(expanded)
    text2 = re.sub(r"\s+", " ", text2).strip()
    return text2


def normalize_column_name(col_name: str) -> str:
    """
    Final normalized column name candidate.
    """
    s = clean_column_name(col_name)
    s = apply_synonyms(s)

    # Optional phrase cleanup
    phrase_map = {
        "customer number": "customer id",
        "customer no": "customer id",
        "item no": "item number",
        "part no": "part number",
        "manufacturer no": "manufacturer number",
        "manufacturer pn": "manufacturer part number",
        "unit measure": "unit of measure",
    }

    for old, new in phrase_map.items():
        if s == old:
            s = new

    s = re.sub(r"\s+", " ", s).strip()
    return s


def token_set(text: str) -> set:
    return set(text.split()) if text else set()


def similarity_score(a: str, b: str) -> float:
    """
    Combined similarity score:
    - exact match
    - token overlap (Jaccard)
    - string similarity
    """
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0

    ta = token_set(a)
    tb = token_set(b)

    jaccard = len(ta & tb) / len(ta | tb) if (ta | tb) else 0.0
    seq = SequenceMatcher(None, a, b).ratio()

    # Give more weight to token overlap
    score = 0.65 * jaccard + 0.35 * seq
    return score


# =========================
# Excel reading helpers
# =========================
def get_excel_engine(file_path: Path):
    """
    Pick engine based on file extension.
    """
    suffix = file_path.suffix.lower()

    if suffix == ".xlsb":
        return "pyxlsb"
    elif suffix in {".xlsx", ".xlsm"}:
        return "openpyxl"
    elif suffix == ".xls":
        # Requires xlrd installed
        return "xlrd"
    else:
        return None


def read_sheet_headers(file_path: Path, sheet_name: str):
    """
    Read only header row from a sheet.
    """
    engine = get_excel_engine(file_path)
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        engine=engine,
        nrows=0
    )
    return list(df.columns)


def get_sheet_names(file_path: Path):
    """
    Get all sheet names from an Excel file.
    """
    engine = get_excel_engine(file_path)
    xls = pd.ExcelFile(file_path, engine=engine)
    return xls.sheet_names


# =========================
# Grouping / standard name inference
# =========================
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


def infer_standard_names(normalized_names, threshold=0.82):
    """
    Group similar normalized names across all files/sheets and assign
    a representative 'standard column name' to each normalized name.
    """
    unique_names = sorted(set(x for x in normalized_names if x))
    if not unique_names:
        return {}

    uf = UnionFind(len(unique_names))

    # Pairwise grouping
    for i in range(len(unique_names)):
        for j in range(i + 1, len(unique_names)):
            score = similarity_score(unique_names[i], unique_names[j])
            if score >= threshold:
                uf.union(i, j)

    # Build groups
    groups = defaultdict(list)
    for i, name in enumerate(unique_names):
        root = uf.find(i)
        groups[root].append(name)

    # Representative choice:
    # choose the most frequent name globally, tie-break by shortest length, then alpha
    freq = Counter(normalized_names)
    mapping = {}

    for members in groups.values():
        representative = sorted(
            members,
            key=lambda x: (-freq[x], len(x), x)
        )[0]

        for member in members:
            mapping[member] = representative

    return mapping


# =========================
# Main logic
# =========================
def build_excel_metadata(input_folder: str, output_file: str):
    input_path = Path(input_folder)

    files = [
        f for f in input_path.iterdir()
        if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    all_rows = []
    all_normalized_names = []

    for file_path in files:
        try:
            sheet_names = get_sheet_names(file_path)
        except Exception as e:
            print(f"[ERROR] Could not open file: {file_path.name} | {e}")
            continue

        for sheet_name in sheet_names:
            try:
                headers = read_sheet_headers(file_path, sheet_name)
            except Exception as e:
                print(f"[ERROR] Could not read sheet: {file_path.name} | {sheet_name} | {e}")
                continue

            for col in headers:
                raw_col = "" if pd.isna(col) else str(col)
                normalized_col = normalize_column_name(raw_col)

                all_rows.append({
                    "File Name": file_path.name,
                    "Tab Name": sheet_name,
                    "Column Name": raw_col,
                    "Normalized Column Name": normalized_col,  # temporary helper
                })

                if normalized_col:
                    all_normalized_names.append(normalized_col)

    # Infer global standard names
    standard_name_map = infer_standard_names(all_normalized_names, threshold=0.82)

    # Final output rows
    for row in all_rows:
        normalized_col = row["Normalized Column Name"]
        row["Standard Column Name"] = standard_name_map.get(normalized_col, normalized_col)

    output_df = pd.DataFrame(all_rows)[
        ["File Name", "Tab Name", "Column Name", "Standard Column Name"]
    ]

    # Save result
    output_df.to_excel(output_file, index=False)
    print(f"Done. Output saved to: {output_file}")
    print(f"Total files processed: {len(files)}")
    print(f"Total column records: {len(output_df)}")


if __name__ == "__main__":
    build_excel_metadata(INPUT_FOLDER, OUTPUT_FILE)

c:\Users\Xing.Shen\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: Analysis!$A:$U.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
c:\Users\Xing.Shen\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Sum by cat'!$A:$P.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
c:\Users\Xing.Shen\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: Analysis!$A:$U.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
c:\Users\Xing.Shen\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Sum by cat'!$A:$P.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


Done. Output saved to: excel_column_metadata_output.xlsx
Total files processed: 17
Total column records: 671


In [7]:
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from difflib import SequenceMatcher


# =========================
# Config
# =========================
INPUT_FOLDER = r"Files"   # <-- change this
OUTPUT_FILE = r"excel_column_metadata_output.xlsx"  # <-- change this

SUPPORTED_EXTENSIONS = {".xlsx", ".xlsm", ".xlsb", ".xls"}

# Optional: expand or adjust synonym mappings for your business context
SYNONYM_MAP = {
    "qty": "quantity",
    "q'ty": "quantity",
    "uom": "unit of measure",
    "unitofmeasure": "unit of measure",
    "unitmeasure": "unit of measure",
    "desc": "description",
    "descr": "description",
    "prod": "product",
    "prd": "product",
    "cust": "customer",
    "mfr": "manufacturer",
    "manuf": "manufacturer",
    "mnf": "manufacturer",
    "mpn": "manufacturer part number",
    "pn": "part number",
    "itemno": "item number",
    "itm": "item",
    "amt": "amount",
    "addr": "address",
    "dt": "date",
    "num": "number",
    "#": "number",
    "id": "id",
    "sku": "sku",
    "po": "purchase order",
    "gl": "general ledger",
    "dept": "department",
    "cat": "category",
    "subcat": "subcategory",
    "pkg": "packaging",
    "ea": "each",
}


# =========================
# Text normalization helpers
# =========================
def clean_column_name(col_name: str) -> str:
    """
    Basic cleanup of a raw column name into a normalized text string.
    """
    if pd.isna(col_name):
        return ""

    s = str(col_name).strip().lower()

    # Replace separators with spaces
    s = re.sub(r"[_\-/\\|]+", " ", s)

    # Split camelCase / PascalCase
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", s)

    # Keep alphanumeric and spaces
    s = re.sub(r"[^a-z0-9\s#]", " ", s)

    # Collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    return s


def apply_synonyms(text: str) -> str:
    """
    Replace token-level known abbreviations / common aliases.
    """
    if not text:
        return ""

    tokens = text.split()
    expanded = []

    for token in tokens:
        expanded_value = SYNONYM_MAP.get(token, token)
        expanded.append(expanded_value)

    text2 = " ".join(expanded)
    text2 = re.sub(r"\s+", " ", text2).strip()
    return text2


def normalize_column_name(col_name: str) -> str:
    """
    Final normalized column name candidate.
    """
    s = clean_column_name(col_name)
    s = apply_synonyms(s)

    phrase_map = {
        "customer number": "customer id",
        "customer no": "customer id",
        "item no": "item number",
        "part no": "part number",
        "manufacturer no": "manufacturer number",
        "manufacturer pn": "manufacturer part number",
        "unit measure": "unit of measure",
    }

    for old, new in phrase_map.items():
        if s == old:
            s = new

    s = re.sub(r"\s+", " ", s).strip()
    return s


def token_set(text: str) -> set:
    return set(text.split()) if text else set()


def similarity_score(a: str, b: str) -> float:
    """
    Combined similarity score:
    - exact match
    - token overlap (Jaccard)
    - string similarity
    """
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0

    ta = token_set(a)
    tb = token_set(b)

    jaccard = len(ta & tb) / len(ta | tb) if (ta | tb) else 0.0
    seq = SequenceMatcher(None, a, b).ratio()

    score = 0.65 * jaccard + 0.35 * seq
    return score


# =========================
# Excel reading helpers
# =========================
def get_excel_engine(file_path: Path):
    """
    Pick engine based on file extension.
    """
    suffix = file_path.suffix.lower()

    if suffix == ".xlsb":
        return "pyxlsb"
    elif suffix in {".xlsx", ".xlsm"}:
        return "openpyxl"
    elif suffix == ".xls":
        return "xlrd"
    else:
        return None


def is_blank_cell(value) -> bool:
    """
    Return True if a cell is blank / null / empty string.
    """
    if pd.isna(value):
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def read_sheet_headers(file_path: Path, sheet_name: str):
    """
    Read header row from a sheet.

    Rule:
    - If the first cell of the first row is blank/null/empty, use the second row as header.
    - Otherwise use the first row as header.
    """
    engine = get_excel_engine(file_path)

    # Read first 2 rows with no header so we can inspect raw cell values
    preview_df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        engine=engine,
        header=None,
        nrows=2
    )

    if preview_df.empty:
        return []

    first_cell = preview_df.iloc[0, 0] if preview_df.shape[0] >= 1 and preview_df.shape[1] >= 1 else None

    # If first cell in first row is blank, use second row as header
    if is_blank_cell(first_cell):
        if preview_df.shape[0] < 2:
            return []
        header_row_idx = 1
    else:
        header_row_idx = 0

    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        engine=engine,
        header=header_row_idx,
        nrows=0
    )

    return list(df.columns)


def get_sheet_names(file_path: Path):
    """
    Get all sheet names from an Excel file.
    """
    engine = get_excel_engine(file_path)
    xls = pd.ExcelFile(file_path, engine=engine)
    return xls.sheet_names


# =========================
# Grouping / standard name inference
# =========================
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


def infer_standard_names(normalized_names, threshold=0.82):
    """
    Group similar normalized names across all files/sheets and assign
    a representative 'standard column name' to each normalized name.
    """
    unique_names = sorted(set(x for x in normalized_names if x))
    if not unique_names:
        return {}

    uf = UnionFind(len(unique_names))

    for i in range(len(unique_names)):
        for j in range(i + 1, len(unique_names)):
            score = similarity_score(unique_names[i], unique_names[j])
            if score >= threshold:
                uf.union(i, j)

    groups = defaultdict(list)
    for i, name in enumerate(unique_names):
        root = uf.find(i)
        groups[root].append(name)

    freq = Counter(normalized_names)
    mapping = {}

    for members in groups.values():
        representative = sorted(
            members,
            key=lambda x: (-freq[x], len(x), x)
        )[0]

        for member in members:
            mapping[member] = representative

    return mapping


# =========================
# Main logic
# =========================
def build_excel_metadata(input_folder: str, output_file: str):
    input_path = Path(input_folder)

    files = [
        f for f in input_path.iterdir()
        if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    all_rows = []
    all_normalized_names = []

    for file_path in files:
        try:
            sheet_names = get_sheet_names(file_path)
        except Exception as e:
            print(f"[ERROR] Could not open file: {file_path.name} | {e}")
            continue

        for sheet_name in sheet_names:
            try:
                headers = read_sheet_headers(file_path, sheet_name)
            except Exception as e:
                print(f"[ERROR] Could not read sheet: {file_path.name} | {sheet_name} | {e}")
                continue

            for col in headers:
                raw_col = "" if pd.isna(col) else str(col)
                normalized_col = normalize_column_name(raw_col)

                all_rows.append({
                    "File Name": file_path.name,
                    "Tab Name": sheet_name,
                    "Column Name": raw_col,
                    "Normalized Column Name": normalized_col,  # temporary helper
                })

                if normalized_col:
                    all_normalized_names.append(normalized_col)

    standard_name_map = infer_standard_names(all_normalized_names, threshold=0.82)

    for row in all_rows:
        normalized_col = row["Normalized Column Name"]
        row["Standard Column Name"] = standard_name_map.get(normalized_col, normalized_col)

    output_df = pd.DataFrame(all_rows)[
        ["File Name", "Tab Name", "Column Name", "Standard Column Name"]
    ]

    output_df.to_excel(output_file, index=False)
    print(f"Done. Output saved to: {output_file}")
    print(f"Total files processed: {len(files)}")
    print(f"Total column records: {len(output_df)}")


if __name__ == "__main__":
    build_excel_metadata(INPUT_FOLDER, OUTPUT_FILE)

c:\Users\Xing.Shen\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: Analysis!$A:$U.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
c:\Users\Xing.Shen\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Sum by cat'!$A:$P.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
c:\Users\Xing.Shen\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: Analysis!$A:$U.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
c:\Users\Xing.Shen\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Sum by cat'!$A:$P.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


Done. Output saved to: excel_column_metadata_output.xlsx
Total files processed: 17
Total column records: 899
